# Settimana 7 — Galton e le altezze di genitori e figli: distribuzioni congiunte, covarianza e correlazione

Questo è il notebook della Settimana 7 del corso di Probabilità e Statistica. Contiene gli script Python della [dispensa della settimana](https://qwertyjacob.github.io/PSI-VA-2627/settimana_07/), nello stesso ordine in cui compaiono: ogni script è una cella di codice, preceduta dal titolo della sezione in cui si trova. Le spiegazioni restano nella dispensa.

Per eseguire una cella, premi il pulsante ▶ alla sua sinistra. Gli script sono indipendenti l'uno dall'altro, quindi puoi eseguirli in qualsiasi ordine e modificarli liberamente. Colab non salva le modifiche nel notebook originale: se vuoi conservarle, scegli *File → Salva una copia in Drive*.

## Laboratorio: dalla tabella alle marginali

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_07/#lab-tabella)

In [ ]:
import numpy as np

# Errori nei due moduli (Baron, Esempio 3.6).
# Righe: x = errori nel primo modulo (0, 1).
# Colonne: y = errori nel secondo modulo (0, 1, 2, 3).
P = np.array([[0.20, 0.20, 0.05, 0.05],
              [0.20, 0.10, 0.10, 0.10]])
xs = np.array([0, 1])
ys = np.array([0, 1, 2, 3])
print("Somma di tutte le caselle:", round(P.sum(), 10))

# Marginali: somme lungo le righe e lungo le colonne.
p_x = P.sum(axis=1)   # ogni riga ha x fisso   -> P(X = x)
p_y = P.sum(axis=0)   # ogni colonna ha y fisso -> P(Y = y)
print("P(X = x):", p_x)
print("P(Y = y):", p_y)

# Indipendenza: la congiunta deve coincidere, casella per
# casella, con la tabella dei prodotti dei margini.
prodotti = np.outer(p_x, p_y)
print("Prodotti dei margini:\n", prodotti)
print("Indipendenti?", np.allclose(P, prodotti))

# Z = X + Y: si sommano le caselle con lo stesso x + y.
p_z = np.zeros(len(xs) + len(ys) - 1)
for i, x in enumerate(xs):
    for j, y in enumerate(ys):
        p_z[x + y] += P[i, j]
print("P(Z = z) per z = 0, ..., 4:", p_z.round(2))

# Valori attesi con la regola LOTUS a due variabili.
e_x = (xs * p_x).sum()
e_y = (ys * p_y).sum()
e_xy = sum(x * y * P[i, j]
           for i, x in enumerate(xs)
           for j, y in enumerate(ys))
print(f"E[X] = {e_x:.3f}   E[Y] = {e_y:.3f}   "
      f"E[X + Y] = {(np.arange(5) * p_z).sum():.3f}")
print(f"E[XY] = {e_xy:.3f}   E[X] E[Y] = {e_x * e_y:.3f}")


# Dal notebook 4.3: tre variabili con 5 valori ciascuna.
def joint(x, y, z):
    return 1 / 125   # 5 * 5 * 5 caselle equiprobabili


def marginal_x(x):
    return sum(joint(x, y, z) for y in range(5) for z in range(5))


print("P(X = 0) =", round(marginal_x(0), 10))
print("Caselle con 10 variabili da 5 valori:", 5 ** 10)

## Laboratorio: la tabella di Galton

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_07/#lab-galton)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# La tabella di Galton (1886): altezze in pollici.
# Righe: genitore medio; colonne: figlio adulto.
# Le classi estreme sono aperte ("meno di", "più di").
genitore = np.array([64.0, 64.5, 65.5, 66.5, 67.5, 68.5,
                     69.5, 70.5, 71.5, 72.5, 73.0])
figlio = np.array([61.7, 62.2, 63.2, 64.2, 65.2, 66.2, 67.2,
                   68.2, 69.2, 70.2, 71.2, 72.2, 73.2, 73.7])
N = np.array([
    [1, 0, 2, 4, 1, 2, 2, 1, 1, 0, 0, 0, 0, 0],
    [1, 1, 4, 4, 1, 5, 5, 0, 2, 0, 0, 0, 0, 0],
    [1, 0, 9, 5, 7, 11, 11, 7, 7, 5, 2, 1, 0, 0],
    [0, 3, 3, 5, 2, 17, 17, 14, 13, 4, 0, 0, 0, 0],
    [0, 3, 5, 14, 15, 36, 38, 28, 38, 19, 11, 4, 0, 0],
    [1, 0, 7, 11, 16, 25, 31, 34, 48, 21, 18, 4, 3, 0],
    [0, 0, 1, 16, 4, 17, 27, 20, 33, 25, 20, 11, 4, 5],
    [1, 0, 1, 0, 1, 1, 3, 12, 18, 14, 7, 4, 3, 3],
    [0, 0, 0, 0, 1, 3, 4, 3, 5, 10, 4, 9, 2, 2],
    [0, 0, 0, 0, 0, 0, 0, 1, 2, 1, 2, 7, 2, 4],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 3, 0]])
print("Figli in tutto:", N.sum())

P = N / N.sum()           # frequenze relative: la congiunta
p_gen = P.sum(axis=1)     # marginale del genitore medio
p_fig = P.sum(axis=0)     # marginale del figlio


def riassunto(g, f):
    """Medie, deviazioni standard, covarianza e correlazione."""
    mu_g = (g * p_gen).sum()
    mu_f = (f * p_fig).sum()
    sd_g = np.sqrt(((g - mu_g) ** 2 * p_gen).sum())
    sd_f = np.sqrt(((f - mu_f) ** 2 * p_fig).sum())
    # Covarianza: in ogni casella il prodotto degli scarti,
    # pesato con la frequenza della casella, poi la somma.
    cov = (np.outer(g - mu_g, f - mu_f) * P).sum()
    return mu_g, mu_f, sd_g, sd_f, cov, cov / (sd_g * sd_f)


for nome, k in [("pollici", 1.0), ("centimetri", 2.54)]:
    mu_g, mu_f, sd_g, sd_f, cov, r = riassunto(k * genitore,
                                               k * figlio)
    print(f"In {nome}: medie {mu_g:.1f} e {mu_f:.1f}, "
          f"sigma {sd_g:.2f} e {sd_f:.2f}")
    print(f"    Cov = {cov:.3f}    r = {r:.3f}")

# Regressione verso la media: per ogni classe di genitori,
# l'altezza media dei figli (in centimetri).
print("genitore (cm)  media dei figli (cm)  figli")
for g, riga in zip(genitore, N):
    media = (riga * figlio).sum() / riga.sum()
    print(f"{2.54 * g:13.1f}  {2.54 * media:20.1f}  {riga.sum():5d}")

# Nuvola di punti: un punto per figlio, spostato a caso
# dentro la sua classe di un pollice per non sovrapporsi.
rng = np.random.default_rng(1886)
righe, colonne = np.nonzero(N)
quanti = N[righe, colonne]
x = np.repeat(genitore[righe], quanti)
y = np.repeat(figlio[colonne], quanti)
x = 2.54 * (x + rng.uniform(-0.5, 0.5, x.size))
y = 2.54 * (y + rng.uniform(-0.5, 0.5, y.size))
plt.scatter(x, y, s=6, alpha=0.4)
plt.axvline(x.mean(), ls="--", c="gray")
plt.axhline(y.mean(), ls="--", c="gray")
plt.xlabel("Genitore medio (cm)")
plt.ylabel("Figlio (cm)")
plt.title(f"r dei punti = {np.corrcoef(x, y)[0, 1]:.2f}")
plt.tight_layout()
plt.show()

## Laboratorio: quando la correlazione inganna

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_07/#lab-inganni)

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
N = 10_000

# 1) Forma a U: Y dipende da X quasi senza rumore.
x = rng.uniform(-1, 1, N)
y = x ** 2 + rng.normal(0, 0.05, N)
print(f"Forma a U: r = {np.corrcoef(x, y)[0, 1]:+.3f}")
# Eppure conoscere X dice moltissimo su Y.
print(f"  media di Y quando |X| < 0,2: {y[np.abs(x) < 0.2].mean():.3f}")
print(f"  media di Y quando |X| > 0,8: {y[np.abs(x) > 0.8].mean():.3f}")

# 2) Un confondente. La dimensione di un modulo (in migliaia
# di righe) fa crescere sia i commenti TODO sia i bug, ma i
# TODO non causano bug: nel modello i bug dipendono solo
# dalla dimensione.
righe = rng.uniform(1, 20, N)
todo = rng.poisson(0.8 * righe)
bug = rng.poisson(0.5 * righe)
r_tutti = np.corrcoef(todo, bug)[0, 1]
print(f"TODO e bug, tutti i moduli:       r = {r_tutti:+.3f}")

# Confrontiamo solo moduli di dimensione quasi uguale:
# la dimensione è (quasi) fissa e il legame sparisce.
simili = (righe > 9.5) & (righe < 10.5)
r_simili = np.corrcoef(todo[simili], bug[simili])[0, 1]
print(f"TODO e bug, moduli da circa 10 mila righe "
      f"({simili.sum()} moduli): r = {r_simili:+.3f}")